# Vesuvius V11 - Advanced Inference & Submission

## Based on Competition Discussion Insights:
- **Host recommendations**: 2D slicewise skeletonization, Frangi filter, merge detection
- **hengck23's approach**: Endpoint detection + path tracing for hole filling
- **Level 1-3 processing strategies**

## Post-Processing Pipeline:
1. Threshold + small component removal
2. Frangi/Sato filter enhancement (sheetness)
3. Surface-aware SDF smoothing
4. Slice-wise hole filling (all 3 axes)
5. Endpoint-based hole tracing
6. Topology validation

In [ ]:
!pip install imagecodecs tifffile scikit-image -q

In [ ]:
# =============================================================================
# IMPORTS & CONFIG
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import warnings
from pathlib import Path
from typing import List, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast

from scipy import ndimage
from scipy.ndimage import (
    binary_fill_holes, distance_transform_edt, gaussian_filter,
    label, generate_binary_structure, binary_dilation, binary_erosion
)
from skimage.morphology import skeletonize
from skimage.filters import frangi

warnings.filterwarnings('ignore')

@dataclass
class Config:
    # Paths
    TEST_ROOT: Path = Path("/kaggle/input/vesuvius-challenge-2025/test")
    CHECKPOINT_PATH: Path = Path("/kaggle/input/v11-checkpoint/fold_0/best_model.pth")
    OUTPUT_DIR: Path = Path("/kaggle/working")
    
    # Model
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = field(default_factory=lambda: [32, 64, 128, 256, 320, 320])
    N_BLOCKS: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 6, 6])
    
    # Inference
    OVERLAP: float = 0.5
    TTA_LEVEL: str = "flip"
    USE_BFLOAT16: bool = True
    
    # Post-processing
    THRESHOLD: float = 0.5
    USE_FRANGI: bool = True
    USE_HOLE_TRACING: bool = True
    MIN_COMPONENT_SIZE: int = 100
    SURFACE_SIGMA: float = 0.5
    MAX_GAP_DISTANCE: int = 15
    
    DEVICE: str = "cuda"

cfg = Config()
print(f"Checkpoint: {cfg.CHECKPOINT_PATH}")
print(f"Post-processing: Frangi={cfg.USE_FRANGI}, HoleTracing={cfg.USE_HOLE_TRACING}")

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE (same as training)
# =============================================================================

def get_num_groups(channels, max_groups=32):
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1

class HybridConv3d(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)
    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True))
    def forward(self, x): return self.conv(x)

class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList([ConvBlock(self.width, self.width, use_hybrid) for _ in range(scales-1)])
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)
    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i+1] + outputs[-1]) if i > 0 else conv(splits[i+1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))

class AttentionBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)
    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(self.conv_spatial(torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)))
        return x_ca * sa

class SurfaceRefinementBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True))
    def forward(self, x):
        return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))

class TopoPreservingUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, 2, use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i]*2, features[i]))
            else:
                self.dec_convs.append(nn.Sequential(
                    ConvBlock(features[i]*2, features[i], use_hybrid_conv),
                    MultiScaleResBlock(features[i], 2, use_hybrid_conv)))
        self.final = nn.Conv3d(features[0], out_ch, 1)
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        enc_features = enc_features[::-1]
        x = enc_features[0]
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.final(x)

print("Model loaded")

In [ ]:
# =============================================================================
# INFERENCE FUNCTIONS
# =============================================================================

def robust_zscore_normalize(img, lower_pct=0.5, upper_pct=99.5):
    p_low, p_high = np.percentile(img, [lower_pct, upper_pct])
    img_clipped = np.clip(img, p_low, p_high)
    return ((img_clipped - img_clipped.mean()) / (img_clipped.std() + 1e-8)).astype(np.float32)

def create_gaussian_weight(patch_size, sigma=0.125):
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d/2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h/2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w/2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)

@torch.no_grad()
def sliding_window_inference(model, volume, patch_size, overlap=0.5, device='cuda', use_bf16=True):
    model.eval()
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd*(1-overlap)), int(ph*(1-overlap)), int(pw*(1-overlap))
    
    pad_d, pad_h, pad_w = max(0, pd-D), max(0, ph-H), max(0, pw-W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume = np.pad(volume, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
        D, H, W = volume.shape
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)
    
    z_pos = list(range(0, max(1, D-pd+1), sd))
    if z_pos[-1] + pd < D: z_pos.append(D - pd)
    y_pos = list(range(0, max(1, H-ph+1), sh))
    if y_pos[-1] + ph < H: y_pos.append(H - ph)
    x_pos = list(range(0, max(1, W-pw+1), sw))
    if x_pos[-1] + pw < W: x_pos.append(W - pw)
    
    vol_norm = robust_zscore_normalize(volume)
    
    for z in tqdm(z_pos, desc="Sliding window", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw]
                inp = torch.from_numpy(patch[None, None]).to(device, dtype=dtype)
                with autocast(device_type='cuda', dtype=dtype):
                    prob = torch.sigmoid(model(inp)).squeeze().float().cpu().numpy()
                pred_sum[z:z+pd, y:y+ph, x:x+pw] += prob * gauss
                weight_sum[z:z+pd, y:y+ph, x:x+pw] += gauss
    
    pred = pred_sum / np.maximum(weight_sum, 1e-8)
    if pad_d > 0: pred = pred[:-pad_d]
    if pad_h > 0: pred = pred[:, :-pad_h]
    if pad_w > 0: pred = pred[:, :, :-pad_w]
    return pred

@torch.no_grad()
def inference_with_tta(model, volume, patch_size, overlap=0.5, device='cuda', use_bf16=True, tta='flip'):
    preds = [sliding_window_inference(model, volume, patch_size, overlap, device, use_bf16)]
    if tta in ['flip', 'full']:
        for axis in [0, 1, 2]:
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference(model, vol_flip, patch_size, overlap, device, use_bf16)
            preds.append(np.flip(pred_flip, axis).copy())
    if tta == 'full':
        for k in [1, 2, 3]:
            vol_rot = np.rot90(volume, k=k, axes=(1, 2)).copy()
            pred_rot = sliding_window_inference(model, vol_rot, patch_size, overlap, device, use_bf16)
            preds.append(np.rot90(pred_rot, k=-k, axes=(1, 2)).copy())
    return np.mean(preds, axis=0)

print("Inference functions loaded")

In [ ]:
# =============================================================================
# ADVANCED POST-PROCESSING (From Discussion Insights)
# =============================================================================

def count_components(mask):
    struct = generate_binary_structure(3, 3)
    _, n = label(mask, structure=struct)
    return n

def remove_small_components(mask, min_size=100):
    struct = generate_binary_structure(3, 3)
    labeled, n = label(mask, structure=struct)
    if n == 0: return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False
    result = mask.copy()
    result[small[labeled]] = 0
    return result

def surface_smoothing(mask, sigma=0.5):
    """SDF-based surface smoothing."""
    dist_in = distance_transform_edt(mask)
    dist_out = distance_transform_edt(~mask)
    sdf = gaussian_filter((dist_in - dist_out).astype(np.float32), sigma=sigma)
    return (sdf > 0).astype(np.uint8)

def slice_wise_hole_fill(mask):
    """Fill holes in 2D slices (catches tunnels)."""
    filled = mask.copy()
    for axis in [0, 1, 2]:
        for i in range(mask.shape[axis]):
            if axis == 0:
                filled[i] = binary_fill_holes(filled[i])
            elif axis == 1:
                filled[:, i, :] = binary_fill_holes(filled[:, i, :])
            else:
                filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled

def skeletonize_2d_slicewise(mask):
    """2D slicewise skeletonization (Host recommended)."""
    skeleton = np.zeros_like(mask, dtype=np.uint8)
    for i in range(mask.shape[0]):
        skeleton[i] = skeletonize(mask[i] > 0).astype(np.uint8)
    return skeleton

def detect_endpoints(skeleton):
    """Detect endpoints (voxels with 1 neighbor)."""
    kernel = np.ones((3, 3, 3), dtype=np.int32)
    kernel[1, 1, 1] = 0
    neighbors = ndimage.convolve(skeleton.astype(np.int32), kernel, mode='constant')
    return ((skeleton > 0) & (neighbors == 1)).astype(np.uint8)

def find_endpoint_pairs(endpoints, max_distance=20):
    """Find pairs of endpoints to connect."""
    coords = np.argwhere(endpoints > 0)
    if len(coords) < 2: return []
    pairs, used = [], set()
    for i, c1 in enumerate(coords):
        if i in used: continue
        min_dist, best_j = float('inf'), -1
        for j, c2 in enumerate(coords):
            if j <= i or j in used: continue
            dist = np.linalg.norm(c1 - c2)
            if dist < min_dist and dist < max_distance:
                min_dist, best_j = dist, j
        if best_j >= 0:
            pairs.append((tuple(c1), tuple(coords[best_j])))
            used.update([i, best_j])
    return pairs

def trace_path(prob_map, start, end):
    """Trace path between two points following high probability."""
    path = [start]
    current = np.array(start, dtype=np.float32)
    target = np.array(end, dtype=np.float32)
    max_steps = int(np.linalg.norm(target - current) * 3)
    
    for _ in range(max_steps):
        if np.linalg.norm(current - target) < 2:
            path.append(tuple(target.astype(int)))
            break
        direction = (target - current) / (np.linalg.norm(target - current) + 1e-8)
        best_next = current + direction
        best_prob = -1
        for offset in np.random.randn(5, 3) * 0.5:
            cand = np.clip(current + direction + offset, 0, np.array(prob_map.shape) - 1)
            try:
                prob = prob_map[int(cand[0]), int(cand[1]), int(cand[2])]
                if prob > best_prob:
                    best_prob, best_next = prob, cand
            except: continue
        current = best_next
        path.append(tuple(current.astype(int)))
    return path

def fill_holes_by_tracing(mask, prob_map, max_gap=15):
    """Fill holes by tracing between endpoints (hengck23's approach)."""
    skeleton = skeletonize_2d_slicewise(mask)
    endpoints = detect_endpoints(skeleton)
    pairs = find_endpoint_pairs(endpoints, max_distance=max_gap)
    
    filled = mask.copy()
    for start, end in pairs:
        path = trace_path(prob_map, start, end)
        for z, y, x in path:
            try:
                filled[max(0,z-1):z+2, max(0,y-1):y+2, max(0,x-1):x+2] = 1
            except: continue
    return filled

def frangi_enhance(mask, sigmas=range(1, 3)):
    """Enhance sheet-like structures with Frangi filter."""
    dist = distance_transform_edt(mask)
    dist_blur = gaussian_filter(dist, sigma=1.0)
    if dist_blur.max() > 0:
        dist_norm = dist_blur / dist_blur.max()
    else:
        return mask
    
    result = np.zeros_like(mask, dtype=np.float32)
    for i in range(mask.shape[0]):
        try:
            result[i] = frangi(dist_norm[i].astype(np.float32), sigmas=sigmas, black_ridges=False)
        except:
            result[i] = dist_norm[i]
    
    thresh = 0.1 * result.max() if result.max() > 0 else 0.5
    return (mask | (result > thresh)).astype(np.uint8)

def topology_safe_op(mask, op_func, **kwargs):
    """Apply operation, revert if components merge."""
    n_before = count_components(mask)
    result = op_func(mask, **kwargs)
    n_after = count_components(result)
    if n_after < n_before:
        print(f"  [REVERT] {op_func.__name__}: {n_before}->{n_after}")
        return mask
    return result

def advanced_postprocess(pred_prob, threshold=0.5, use_frangi=True, use_hole_tracing=True,
                         min_size=100, surface_sigma=0.5, max_gap=15, verbose=True):
    """Full advanced post-processing pipeline."""
    if verbose: print("Advanced Post-processing:")
    
    # 1. Threshold
    mask = (pred_prob > threshold).astype(np.uint8)
    if verbose: print(f"  1. Threshold: FG={100*mask.mean():.2f}%")
    
    # 2. Remove small components
    n1 = count_components(mask)
    mask = remove_small_components(mask, min_size)
    n2 = count_components(mask)
    if verbose: print(f"  2. Remove small: {n1}->{n2} components")
    
    # 3. Frangi enhancement
    if use_frangi:
        try:
            mask = frangi_enhance(mask)
            if verbose: print(f"  3. Frangi enhance: FG={100*mask.mean():.2f}%")
        except Exception as e:
            if verbose: print(f"  3. Frangi skipped: {e}")
    
    # 4. Surface smoothing
    mask = surface_smoothing(mask, sigma=surface_sigma)
    if verbose: print(f"  4. Surface smooth: FG={100*mask.mean():.2f}%")
    
    # 5. Slice-wise hole fill
    mask = topology_safe_op(mask, slice_wise_hole_fill)
    if verbose: print(f"  5. Hole fill: FG={100*mask.mean():.2f}%")
    
    # 6. Endpoint-based hole tracing
    if use_hole_tracing:
        try:
            mask = fill_holes_by_tracing(mask, pred_prob, max_gap=max_gap)
            if verbose: print(f"  6. Hole tracing: FG={100*mask.mean():.2f}%")
        except Exception as e:
            if verbose: print(f"  6. Hole tracing skipped: {e}")
    
    # 7. Final cleanup
    mask = remove_small_components(mask, min_size)
    n_final = count_components(mask)
    if verbose: print(f"  7. Final: {n_final} components, FG={100*mask.mean():.2f}%")
    
    return mask

print("Advanced post-processing loaded")

In [ ]:
# =============================================================================
# RLE ENCODING
# =============================================================================

def rle_encode_3d(mask):
    flat = mask.flatten(order='F')
    if flat.sum() == 0: return ''
    padded = np.concatenate([[0], flat, [0]])
    transitions = np.diff(padded)
    starts = np.where(transitions == 1)[0] + 1
    ends = np.where(transitions == -1)[0] + 1
    lengths = ends - starts
    return ' '.join([f"{s} {l}" for s, l in zip(starts, lengths)])

print("RLE encoding loaded")

In [ ]:
# =============================================================================
# LOAD MODEL
# =============================================================================

def load_model(checkpoint_path, device='cuda'):
    print(f"Loading: {checkpoint_path}")
    model = TopoPreservingUNet3D(features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    print(f"  Epoch: {ckpt.get('epoch', '?')}, Best Dice: {ckpt.get('best_dice', '?'):.4f}")
    return model

model = load_model(cfg.CHECKPOINT_PATH, cfg.DEVICE)

if hasattr(torch, 'compile'):
    print("Compiling model...")
    model = torch.compile(model, mode='reduce-overhead')

In [ ]:
# =============================================================================
# RUN INFERENCE
# =============================================================================

test_files = sorted(cfg.TEST_ROOT.glob("*.tif")) + sorted(cfg.TEST_ROOT.glob("*.tiff"))
print(f"Found {len(test_files)} test volumes")

submissions = []

for vol_path in test_files:
    vol_id = vol_path.stem
    print(f"\n{'='*60}\nProcessing: {vol_id}\n{'='*60}")
    
    volume = tifffile.imread(str(vol_path)).astype(np.float32)
    print(f"Shape: {volume.shape}")
    
    # Inference
    print(f"Running inference (TTA={cfg.TTA_LEVEL})...")
    pred_prob = inference_with_tta(model, volume, cfg.PATCH_SIZE, cfg.OVERLAP,
                                    cfg.DEVICE, cfg.USE_BFLOAT16, cfg.TTA_LEVEL)
    
    # Advanced post-processing
    pred_mask = advanced_postprocess(
        pred_prob, threshold=cfg.THRESHOLD,
        use_frangi=cfg.USE_FRANGI, use_hole_tracing=cfg.USE_HOLE_TRACING,
        min_size=cfg.MIN_COMPONENT_SIZE, surface_sigma=cfg.SURFACE_SIGMA,
        max_gap=cfg.MAX_GAP_DISTANCE
    )
    
    # RLE encode
    rle = rle_encode_3d(pred_mask)
    submissions.append({'id': vol_id, 'rle': rle})
    print(f"RLE length: {len(rle)}")
    
    del volume, pred_prob, pred_mask
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}\nINFERENCE COMPLETE\n{'='*60}")

In [ ]:
# =============================================================================
# CREATE SUBMISSION
# =============================================================================

submission_df = pd.DataFrame(submissions)
submission_df.to_csv(cfg.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission saved: {cfg.OUTPUT_DIR / 'submission.csv'}")
print(f"Samples: {len(submission_df)}")
submission_df.head()